# JDsearch – File 2: Feature engineering

**Input:** the Parquet tables written by the EDA notebook (`Data/Processed`). **Output:** the feature files for the two models, written to `Data/Features`.

| Model | Features | Section |
|---|---|---|
| **A: non-personalized**, f(Q, P) | 7 query–product, popularity and metadata features | 2 |
| **B: personalized**, f(Q, P, H_u) | the same 7 plus 23 history features | 3 |

**Contents**

| Section | What happens |
|---|---|
| 0 | Setup and loading the EDA files |
| 1 | Prepare the data: users and splits, clean products, candidate base table, shared training-only statistics |
| 2 | Model A features |
| 3 | Model B features |
| 4 | Full-data build and final checks |

Most steps are skipped on a rerun if their files already exist; set `FORCE = True` to rebuild. Model training and evaluation are in `Baseline_Model.ipynb`, `Advanced_Model.ipynb` and `ModelComparison.ipynb`.

## 0. Setup and loading
Imports, folder paths and constants. `N_DEV_USERS` is the number of training users used while developing (30k of about 121k keeps every step fast); section 4 repeats the build on all of them. The three windows are the logging lookbacks found in the temporal trace (clicks 7 days, carts 90 days, orders 365 days).

In [ ]:
import os, glob, shutil, time, json
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.compute as pc, pyarrow.parquet as pq

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE  = '/content/drive/MyDrive/bt4222_project'
    LOCAL = '/content/jd_local'
except ImportError:
    BASE, LOCAL = os.environ['JD_BASE'], os.environ['JD_LOCAL']

OUT, FEAT = f'{BASE}/Data/Processed', f'{BASE}/Data/Features'     # read from OUT, write to FEAT
FEAT_LOCAL = f'{LOCAL}/features'                                  # work on the fast local disk, copy results to Drive
for d in (LOCAL, FEAT, FEAT_LOCAL): os.makedirs(d, exist_ok=True)

SEED = 42
W_CLICK, W_CART, W_ORD = 7, 90, 365      # logging windows in days (A4 temporal trace)
N_DEV_USERS = 30_000                     # training users used while developing; None = all
N_FOLDS = 5                              # user folds for the out-of-fold stacker
FORCE = False                            # True = rebuild files that already exist
SPLITS = ['train', 'val', 'test']

Mounted at /content/drive


### 0.1 Copy the input files to the local disk
Copies the products table, the split lists and the per-split test and candidate tables from Drive to the Colab disk (skipped if already there). The history tables (`events*`) are only needed from step 6 onwards.

In [ ]:
def fetch_products(wids, columns, path):
    """Stream a products Parquet file and keep only the requested wids (low RAM)."""
    keep = pa.array(np.unique(np.asarray(wids, dtype=np.int64)), pa.int64())
    cols = ['wid'] + [c for c in columns if c != 'wid']
    parts = []
    for b in pq.ParquetFile(path).iter_batches(batch_size=1_000_000, columns=cols):
        t = pa.Table.from_batches([b])
        t = t.filter(pc.is_in(t['wid'], value_set=keep))
        if t.num_rows: parts.append(t)
    return pa.concat_tables(parts) if parts else pa.table({c: pa.array([], pa.int64()) for c in cols})

def ensure_local(names):
    for n in names:
        dst, src = f'{LOCAL}/{n}.parquet', f'{OUT}/{n}.parquet'
        if os.path.exists(dst) and (not os.path.exists(src) or os.path.getsize(dst) == os.path.getsize(src)): continue
        if not os.path.exists(src): raise FileNotFoundError(f'{src} missing: run the EDA notebook (A1/A2) first')
        t = time.time(); shutil.copyfile(src, dst); print(f'copied {n} ({os.path.getsize(dst)/1e6:,.0f} MB, {time.time()-t:.0f}s)')

ensure_local(['products', 'splits'] + [f'{t}_{s}' for t in ('tests', 'cands') for s in SPLITS])

# feature files: local copy + backup on Drive
def feat_path(name, ext='parquet'): return f'{FEAT_LOCAL}/{name}.{ext}'
def feat_exists(name, ext='parquet'):
    """True if the file exists locally or on Drive (then it is copied back to the local disk)."""
    if os.path.exists(feat_path(name, ext)): return True
    if os.path.exists(f'{FEAT}/{name}.{ext}'): shutil.copyfile(f'{FEAT}/{name}.{ext}', feat_path(name, ext)); return True
    return False
def feat_save(name, ext='parquet'): shutil.copyfile(feat_path(name, ext), f'{FEAT}/{name}.{ext}')
print('input files ready')

input files ready


## 1. Prepare the data
Everything the features are built from. Sections 2 and 3 only add feature columns; they never change the tables built here.

Facts from the EDA and the pre-modelling checks that these steps rely on:
- term ids in queries, product names and `brand_name` are numbers joined by `\x18` and share one vocabulary (mean query coverage 0.72 vs 0.03 for a shuffled control);
- the candidate lists in the raw data are sorted positives-first, so list position is never used and candidates are shuffled;
- the placeholders are three brand ids (rows with a missing brand name) and `shop_id = -1`.

In [ ]:
SEP = '\x18'          # separator between term ids (EDA)
BRAND_TERMS = True    # brand_name is a list of term ids (checked in the first run of this notebook)

### 1.1 User table: usable queries, development sample and folds
One row per user with: its split, `use` = the test query has at least one positive candidate (the 2,103 users without one are dropped from training and evaluation), `in_dev` = part of the development sample (a fixed random draw of `N_DEV_USERS` usable training users, all validation and test users), and `fold` = one of 5 user folds for training users (used later for out-of-fold stacking). Seed 42, so the draw never changes.

In [ ]:
def build_users(force=False):
    if not force and feat_exists('users'): print('users: found, skipping'); return
    sp = pd.read_parquet(f'{LOCAL}/splits.parquet')
    te = pd.concat([pq.read_table(f'{LOCAL}/tests_{s}.parquet', columns=['user_id', 'n_cand', 'n_pos', 'n_hist']).to_pandas()
                    for s in SPLITS])
    u = te.merge(sp, on='user_id', how='left').reset_index(drop=True)
    assert u.split.notna().all() and len(u) == len(sp) == u.user_id.nunique(), 'splits and tests do not line up'
    u['use'] = u.n_pos > 0
    rng = np.random.default_rng(SEED)
    u['in_dev'] = u.split != 'train'
    tr = u.index[u.split == 'train']
    pool = tr[u.loc[tr, 'use'].values]
    pick = rng.choice(pool, min(N_DEV_USERS or len(pool), len(pool)), replace=False)
    u.loc[pick, 'in_dev'] = True
    u['fold'] = -1
    u.loc[tr, 'fold'] = (np.arange(len(tr)) % N_FOLDS)[rng.permutation(len(tr))]
    u['fold'] = u.fold.astype('int8')
    u.to_parquet(feat_path('users'), index=False); feat_save('users')

build_users(FORCE)
users = pd.read_parquet(feat_path('users'))
print(users.groupby('split').agg(users=('user_id', 'size'), usable=('use', 'sum'), in_dev=('in_dev', 'sum')).loc[SPLITS])
print('training folds:', users[users.split == 'train'].fold.value_counts().sort_index().to_dict())

users: found, skipping
        users  usable  in_dev
split                        
train  121681  120174   30000
val     26074   25787   26074
test    26076   25767   26076
training folds: {0: 24337, 1: 24336, 2: 24336, 3: 24336, 4: 24336}


### 1.2 Clean the products
The EDA found two placeholders: three brand ids that stand for "no brand" (all rows without a brand name), and `shop_id = -1`. Both become NaN, and a `has_brand` flag is added. Streams the 12M products in batches and writes `products_clean.parquet`; categories had no placeholder and are untouched.

In [ ]:
def build_products_clean(force=False):
    if not force and feat_exists('products_clean'): print('products_clean: found, skipping'); return
    t0 = time.time()
    b = pq.read_table(f'{LOCAL}/products.parquet', columns=['brand_id', 'brand_name'])
    unk = pc.drop_null(pc.unique(pc.filter(b['brand_id'], pc.is_null(b['brand_name']))))     # brand ids whose name is missing
    in_unk = pc.is_in(b['brand_id'], value_set=unk)
    n_unk, n_named = pc.sum(in_unk).as_py() or 0, pc.sum(pc.and_(in_unk, pc.is_valid(b['brand_name']))).as_py() or 0
    print(f'placeholder brand ids: {unk.to_pylist()} | {n_unk:,} products, of which {n_named:,} have a brand name')
    assert n_unk == 0 or n_named / n_unk < 0.01, 'a "placeholder" id also has named products: inspect before cleaning'
    unk_list = unk.to_pylist(); del b
    writer = None
    for batch in pq.ParquetFile(f'{LOCAL}/products.parquet').iter_batches(batch_size=1_000_000):
        t = pa.Table.from_batches([batch])
        is_unk = pc.is_in(t['brand_id'], value_set=unk)
        t = t.set_column(t.schema.get_field_index('brand_id'), 'brand_id',
                         pc.if_else(is_unk, pa.scalar(None, pa.int64()), t['brand_id']))
        t = t.set_column(t.schema.get_field_index('shop_id'), 'shop_id',
                         pc.if_else(pc.equal(t['shop_id'], -1), pa.scalar(None, pa.int64()), t['shop_id']))
        t = t.append_column('has_brand', pc.invert(is_unk))
        if writer is None: writer = pq.ParquetWriter(feat_path('products_clean'), t.schema, compression='zstd')
        writer.write_table(t)
    writer.close(); feat_save('products_clean')
    print(f'products_clean written in {time.time()-t0:.0f}s')

build_products_clean(FORCE)
pcl = pq.read_table(feat_path('products_clean'), columns=['brand_id', 'shop_id', 'has_brand'])
print(f'{pcl.num_rows:,} products | brand_id missing {pcl["brand_id"].null_count/pcl.num_rows:.2%} | '
      f'shop_id missing {pcl["shop_id"].null_count/pcl.num_rows:.2%} | has_brand {pc.mean(pcl["has_brand"]).as_py():.2%}')
del pcl

products_clean: found, skipping
12,141,247 products | brand_id missing 7.78% | shop_id missing 0.85% | has_brand 92.22%


### 1.3 Candidate base table and term lists
For a set of users, `build_tag` writes three files: `base_<tag>` (one row per user and candidate: `label`, product metadata, `has_meta`, `has_brand`, `n_cand`), `cand_terms_<tag>` (term list of each candidate name, plus brand terms) and `qry_terms_<tag>` (term list of each test query). Only users whose query has a positive are included.

The candidates are shuffled with a fixed seed inside each query and `cand_pos` is dropped, because the raw lists are sorted positives-first. It is built now for the development training users and for validation; section 4 builds the full training set and the test set with the same function.

In [ ]:
META_COLS = ['brand_id', 'shop_id', 'cate_id_1', 'cate_id_2', 'cate_id_3', 'cate_id_4', 'has_brand']

def build_tag(tag, split, user_ids, force=False):
    names = [f'base_{tag}', f'cand_terms_{tag}', f'qry_terms_{tag}']
    if not force and all(feat_exists(n) for n in names): print(f'{tag}: found, skipping'); return
    t0 = time.time()
    uid = pa.array(np.sort(user_ids), pa.int32())
    ca = pq.read_table(f'{LOCAL}/cands_{split}.parquet')
    ca = ca.filter(pc.is_in(ca['user_id'], value_set=uid))
    extra = ['brand_name'] if BRAND_TERMS else []
    prod = fetch_products(pc.unique(ca['wid']).to_numpy(), ['name'] + extra + META_COLS, feat_path('products_clean'))

    # base table: candidates + product metadata
    meta = prod.select(['wid'] + META_COLS).to_pandas()          # ids with NaN become float64: exact below 2**53, never downcast to float32
    meta['has_meta'] = True
    df = ca.to_pandas().merge(meta, on='wid', how='left')
    df['has_meta'] = df.has_meta.eq(True)
    df['has_brand'] = df.has_brand.eq(True)
    df['n_cand'] = df.groupby('user_id').wid.transform('size').astype('int16')
    df = df.iloc[np.random.default_rng(SEED).permutation(len(df))].sort_values('user_id', kind='stable')   # random order inside each query
    df.drop(columns='cand_pos').to_parquet(feat_path(f'base_{tag}'), index=False, compression='zstd')

    # term lists (null stays null)
    split_terms = lambda col: pc.split_pattern(col, pattern=SEP)
    terms = pa.table({'wid': prod['wid'], 'terms': split_terms(prod['name'])})
    if BRAND_TERMS: terms = terms.append_column('brand_terms', split_terms(prod['brand_name']))
    pq.write_table(terms, feat_path(f'cand_terms_{tag}'), compression='zstd')
    te = pq.read_table(f'{LOCAL}/tests_{split}.parquet', columns=['user_id', 'query'])
    te = te.filter(pc.is_in(te['user_id'], value_set=uid))
    pq.write_table(pa.table({'user_id': te['user_id'], 'terms': split_terms(te['query'])}),
                   feat_path(f'qry_terms_{tag}'), compression='zstd')
    for n in names: feat_save(n)
    print(f'{tag}: {len(df):,} candidate rows, {len(uid):,} users, {prod.num_rows:,} distinct products ({time.time()-t0:.0f}s)')
    del ca, prod, meta, df, terms, te

SETS = {
    'train_dev': ('train', users.loc[(users.split == 'train') & users.use & users.in_dev, 'user_id'].values),
    'val':       ('val',   users.loc[(users.split == 'val') & users.use, 'user_id'].values),
}
for tag, (split, ids) in SETS.items(): build_tag(tag, split, ids, FORCE)

train_dev: found, skipping
val: found, skipping


### 1.4 Check the base tables
Integrity checks (exact user sets, no repeated products, every query has a positive, `n_cand` matches the tests table, `cand_pos` gone, first row not systematically positive) and a summary per table. Expected from the EDA: about 0.5% of candidates without metadata, about 89 candidates per query, about 1.7 positives per query, first-row-positive share close to the overall positive rate (about 2%) and not 20%+.

In [ ]:
rows = []
for tag, (split, ids) in SETS.items():
    b = pd.read_parquet(feat_path(f'base_{tag}'))
    print(tag, '- repeated product within a query:', int(b.duplicated(['user_id', 'wid']).sum()), '| cand_pos dropped:', 'cand_pos' not in b.columns,
          '| first row positive:', f'{b.groupby("user_id").label.first().gt(0).mean():.1%}')
    assert set(b.user_id) == set(ids), 'user set differs from the requested users'
    exp = users.set_index('user_id').loc[b.user_id.unique(), 'n_cand']
    assert (b.groupby('user_id').size().loc[exp.index] == exp).all(), 'candidate counts differ from the tests table'
    pos = (b.label > 0).groupby(b.user_id).sum()
    assert (pos > 0).all(), 'a query without a positive slipped in'
    q = pd.read_parquet(feat_path(f'qry_terms_{tag}'))
    rows.append(dict(table=tag, users=b.user_id.nunique(), rows=len(b), cands_per_query=round(len(b) / b.user_id.nunique(), 1),
                     positives_per_query=round(pos.mean(), 2), no_meta=round(1 - b.has_meta.mean(), 4),
                     no_brand=round(1 - b.has_brand.mean(), 4), shop_missing=round(b.shop_id.isna().mean(), 4),
                     queries_without_terms=int((q.terms.map(len) == 0).sum())))
try: from IPython.display import display
except ImportError: display = print
display(pd.DataFrame(rows).set_index('table'))
print('label shares (train_dev):', pd.read_parquet(feat_path('base_train_dev'), columns=['label']).label.value_counts(normalize=True).sort_index().round(4).to_dict())
print('all integrity checks passed')

train_dev - repeated product within a query: 0 | cand_pos dropped: True | first row positive: 3.2%
val - repeated product within a query: 0 | cand_pos dropped: True | first row positive: 3.2%


,users,rows,cands_per_query,positives_per_query,no_meta,no_brand,shop_missing,queries_without_terms
table,,,,,,,,
train_dev,30000,2686779,89.6,1.87,0.0045,0.0577,0.0142,0
val,25787,2294519,89.0,1.85,0.0046,0.0592,0.0141,0


label shares (train_dev): {0: 0.9791, 1: 0.0146, 2: 0.0049, 3: 0.0014}
all integrity checks passed


### 1.5 Shared training-only statistics
Two tables that both models use, computed on **training users only** (val and test queries are scored with counts that never include their own history):

| Table | Content | Used by |
|---|---|---|
| `pop_train` | per product: number of training-user events by type (`n_ord` orders, `n_cart` carts, `n_click` clicks). Follow events are not counted. | `prod_pop_ord`, `prod_pop_cart`, `prod_pop_click` (A and B) |
| `idf_train` | per term id: document frequency and IDF over the names of the development-training candidate products, `idf = ln((N+1)/(df+1)) + 1` | `tfidf_cosine` (A), `pqs_max`, `pqs_decay` (B) |

The observed logging horizons are approximately 7 days for clicks, 90 days for carts and 365 days for orders. Rare events outside these horizons are retained. Counts are computed separately per event type. Clicks are included because popularity over all events was the strongest popularity-only ranking in the checks (NDCG@10 0.340 vs 0.330 carts, 0.284 orders).

In [ ]:
ensure_local(['events_train'])

def build_stats(force=False):
    if not force and feat_exists('pop_train') and feat_exists('idf_train'): print('pop_train, idf_train: found, skipping'); return
    t0 = time.time()
    # --- popularity: events of training users, counted per product and event type
    ev = pq.read_table(f'{LOCAL}/events_train.parquet', columns=['wid', 'etype'])
    g = ev.group_by(['wid', 'etype']).aggregate([([], 'count_all')]).to_pandas()
    g['etype'] = g.etype.astype(int)
    pop = (g[g.etype.isin([1, 2, 3])].pivot_table(index='wid', columns='etype', values='count_all', aggfunc='sum', fill_value=0)
             .rename(columns={1: 'n_click', 2: 'n_cart', 3: 'n_ord'}).reset_index())
    pop.columns.name = None
    for c in ('n_click', 'n_cart', 'n_ord'):
        if c not in pop: pop[c] = 0
    pop = pop[['wid', 'n_ord', 'n_cart', 'n_click']].astype({'wid': 'int64', 'n_ord': 'int32', 'n_cart': 'int32', 'n_click': 'int32'})
    pop.to_parquet(feat_path('pop_train'), index=False, compression='zstd')
    # --- IDF: names of the products shown to the training users (dev sample)
    ct = pq.read_table(feat_path('cand_terms_train_dev'), columns=['terms'])['terms'].combine_chunks()
    n_docs = len(ct) - ct.null_count
    flat, parent = pc.list_flatten(ct), pc.list_parent_indices(ct)
    dd = pa.table({'p': parent, 't': flat}).group_by(['p', 't']).aggregate([])          # each term once per product name
    vc = pc.value_counts(dd['t'])
    idf = pd.DataFrame({'term': vc.field('values').to_pylist(), 'df': vc.field('counts').to_numpy()})
    idf['idf'] = np.log((n_docs + 1) / (idf.df + 1)) + 1
    idf.attrs['n_docs'] = int(n_docs)
    idf.to_parquet(feat_path('idf_train'), index=False, compression='zstd')
    for n in ('pop_train', 'idf_train'): feat_save(n)
    print(f'stats built in {time.time()-t0:.0f}s (N docs for IDF = {n_docs:,})')

build_stats(FORCE)
pop, idf = pd.read_parquet(feat_path('pop_train')), pd.read_parquet(feat_path('idf_train'))
print(f'pop_train: {len(pop):,} products with at least one training event | '
      f'orders {int(pop.n_ord.sum()):,} | carts {int(pop.n_cart.sum()):,} | clicks {int(pop.n_click.sum()):,}')
print(f'idf_train: {len(idf):,} terms | idf range {idf.idf.min():.2f} to {idf.idf.max():.2f} | most common terms:')
print(idf.sort_values('df', ascending=False).head(5).to_string(index=False))

pop_train, idf_train: found, skipping
pop_train: 4,347,495 products with at least one training event | orders 4,790,090 | carts 8,116,301 | clicks 5,832,871
idf_train: 401,563 terms | idf range 1.04 to 14.84 | most common terms:
    term      df      idf
60257419 1972135 1.039921
59449168  539453 2.336236
96496462  538849 2.337356
41787705  508441 2.395442
59865766  504119 2.403979


**Check against the EDA.** Joins the popularity table onto the validation candidates and compares with the checks: with all events, about 32% of positives and 61% of negatives have zero popularity; positives should have clearly higher counts than negatives, and the AUC of the all-event count should be near 0.69.

In [ ]:
from sklearn.metrics import roc_auc_score
b = pd.read_parquet(feat_path('base_val'), columns=['user_id', 'wid', 'label']).merge(pop, on='wid', how='left').fillna({'n_ord': 0, 'n_cart': 0, 'n_click': 0})
b['n_all'] = b.n_ord + b.n_cart + b.n_click
b['pos'] = (b.label > 0).astype(int)
rows = [dict(signal=c, zero_share_pos=round(float((b.loc[b.pos == 1, c] == 0).mean()), 3), zero_share_neg=round(float((b.loc[b.pos == 0, c] == 0).mean()), 3),
             auc=round(float(roc_auc_score(b.pos, b[c])), 3)) for c in ('n_ord', 'n_cart', 'n_click', 'n_all')]
display(pd.DataFrame(rows).set_index('signal'))
print('EDA reference (all events): zero share positives 0.316, negatives 0.609, AUC 0.688')
del b

,zero_share_pos,zero_share_neg,auc
signal,,,
n_ord,0.545,0.800,0.639
n_cart,0.378,0.668,0.676
n_click,0.432,0.726,0.669
n_all,0.316,0.609,0.688


EDA reference (all events): zero share positives 0.316, negatives 0.609, AUC 0.688


### 1.6 Check that everything reached Drive
Lists the files in `Data/Features` with their sizes. Drive uploads can lag; if a file is missing, wait a minute and rerun, and run `drive.flush_and_unmount()` before closing the session.

In [ ]:
for f in sorted(glob.glob(f'{FEAT}/*')): print(f'{os.path.getsize(f)/1e6:9.1f} MB  {os.path.basename(f)}')

     26.4 MB  base_test.parquet
     30.4 MB  base_train_c0.parquet
     30.3 MB  base_train_c1.parquet
     30.5 MB  base_train_c2.parquet
     30.6 MB  base_train_c3.parquet
     30.4 MB  base_train_dev.parquet
     26.2 MB  base_val.parquet
    198.6 MB  cand_terms_train_dev.parquet
    172.4 MB  cand_terms_val.parquet
     37.7 MB  cf_book_0.npz
     37.4 MB  cf_book_1.npz
     37.7 MB  cf_book_2.npz
     37.7 MB  cf_book_3.npz
     37.5 MB  cf_book_4.npz
     46.6 MB  cf_book_full.npz
     22.7 MB  featA_test.parquet
     26.4 MB  featA_train_c0.parquet
     26.2 MB  featA_train_c1.parquet
     26.5 MB  featA_train_c2.parquet
     26.4 MB  featA_train_c3.parquet
     26.2 MB  featA_train_dev.parquet
     22.7 MB  featA_val.parquet
     16.1 MB  featB_aff_train_dev.parquet
     13.9 MB  featB_aff_val.parquet
     16.8 MB  featB_cf_train_dev.parquet
     14.9 MB  featB_cf_val.parquet
     34.9 MB  featB_hist_test.parquet
     40.2 MB  featB_hist_train_c0.parquet
     40.0 MB  featB_

## 2. Model A features: f(Q, P)
Seven features that describe how well a candidate matches the query and how popular it is. None of them uses the shopper's history, so Model A is the non-personalized baseline. Model B receives these same seven columns plus the history block.

| Feature | Definition |
|---|---|
| `query_coverage` | share of the query's unique terms that appear in the candidate name |
| `tfidf_cosine` | cosine between the query and the name as TF-IDF vectors (raw term count x IDF from `idf_train`, L2-normalised) |
| `brand_name_match` | 1 if all brand terms appear in the query, 0 if not, NaN if the product has no brand name |
| `prod_pop_ord`, `prod_pop_cart`, `prod_pop_click` | `log1p` of the product's training-user event counts (`pop_train`); 0 if the product never occurs |
| `has_meta` | 1 if the product exists in the metadata table |

Candidates without product metadata have no name, so their three text features are NaN. Terms that are not in `idf_train` (they never occur in the training candidate names) get the maximum IDF, `ln(N+1)+1`. The output `featA_<tag>` has one row per (user_id, wid) in the same order as `base_<tag>`.

### 2.1 Text-match features
Builds sparse term matrices for the distinct candidate products and for the queries, then scores every (query, candidate) row in chunks. Query and name terms are strings of term ids, so a term is the same object in both.

In [ ]:
import scipy.sparse as sp
from sklearn.preprocessing import normalize

idf_tab = pd.read_parquet(feat_path('idf_train'))
N_DOCS = int(pq.read_metadata(feat_path('cand_terms_train_dev')).num_rows)         # distinct training candidate products
IDF_MAX = float(np.log(N_DOCS + 1) + 1)
IDF_VOCAB = pa.array(idf_tab.term.astype(str).values, pa.string())

def csr_from_lists(col, vocab, n_cols):
    """Sparse term-count matrix from a list<string> column; a term repeated inside one list counts several times."""
    col = col.combine_chunks() if hasattr(col, 'combine_chunks') else col
    lens = pc.fill_null(pc.list_value_length(col), 0).to_numpy(zero_copy_only=False).astype(np.int64)
    ids = pc.index_in(pc.list_flatten(col), value_set=vocab).to_numpy(zero_copy_only=False).astype(np.int64)
    m = sp.csr_matrix((np.ones(len(ids), np.float32), ids, np.concatenate([[0], np.cumsum(lens)])), shape=(len(lens), n_cols))
    m.sum_duplicates()
    return m

def text_features(tag, base, chunk=250_000):
    ct = pq.read_table(feat_path(f'cand_terms_{tag}'))
    qt = pq.read_table(feat_path(f'qry_terms_{tag}'))
    # vocabulary = training IDF terms + terms that only occur here (they get the maximum IDF)
    flat = [pc.list_flatten(c.combine_chunks()) for c in (ct['terms'], ct['brand_terms'], qt['terms'])]
    here = pc.unique(pa.concat_arrays(flat))
    extra = pc.filter(here, pc.invert(pc.is_in(here, value_set=IDF_VOCAB)))
    vocab = pa.concat_arrays([IDF_VOCAB, extra.cast(pa.string())])
    V = len(vocab)
    idf = np.concatenate([idf_tab.idf.values, np.full(len(extra), IDF_MAX)]).astype(np.float32)
    P  = csr_from_lists(ct['terms'], vocab, V)                  # candidate name term counts
    Bm = csr_from_lists(ct['brand_terms'], vocab, V)            # candidate brand term counts
    Q  = csr_from_lists(qt['terms'], vocab, V)                  # query term counts
    Pb, Bb, Qb = (m.copy() for m in (P, Bm, Q))
    for m in (Pb, Bb, Qb): m.data[:] = 1
    Pt, Qt = normalize(P.multiply(idf).tocsr()), normalize(Q.multiply(idf).tocsr())
    n_brand, n_q = np.asarray(Bb.sum(1)).ravel(), np.asarray(Qb.sum(1)).ravel()
    pi_all = pd.Index(ct['wid'].to_numpy()).get_indexer(base.wid.values)
    ui_all = pd.Index(qt['user_id'].to_numpy()).get_indexer(base.user_id.values)
    assert (ui_all >= 0).all(), 'a query has no term list'
    known = pi_all >= 0                                        # products without metadata have no name: their text features stay NaN
    pi_all = np.where(known, pi_all, 0)
    cov, cos, bm = (np.full(len(base), np.nan, np.float32) for _ in range(3))
    dot = lambda A, B: np.asarray(A.multiply(B).sum(1)).ravel()
    for a in range(0, len(base), chunk):
        sl = slice(a, a + chunk); pi, ui = pi_all[sl], ui_all[sl]
        ov = dot(Qb[ui], Pb[pi])
        cov[sl] = np.where(n_q[ui] > 0, ov / np.maximum(n_q[ui], 1), np.nan)
        cos[sl] = dot(Qt[ui], Pt[pi])
        bov = dot(Qb[ui], Bb[pi])
        bm[sl] = np.where(n_brand[pi] > 0, (bov == n_brand[pi]).astype(np.float32), np.nan)
    for arr in (cov, cos, bm): arr[~known] = np.nan
    return pd.DataFrame({'query_coverage': cov, 'tfidf_cosine': cos, 'brand_name_match': bm})

### 2.2 Popularity and metadata features, and assembly
Joins the popularity table on `wid` (a product without any training event gets count 0, which is a real zero) and adds `has_meta`. Writes `featA_<tag>` and checks that it lines up row by row with `base_<tag>`, since Model B is joined on (user_id, wid) later.

In [ ]:
POP = pd.read_parquet(feat_path('pop_train')).set_index('wid')
FEAT_A = ['query_coverage', 'tfidf_cosine', 'brand_name_match', 'prod_pop_ord', 'prod_pop_cart', 'prod_pop_click', 'has_meta']

def build_featA(tag, force=False):
    if not force and feat_exists(f'featA_{tag}'): print(f'featA_{tag}: found, skipping'); return
    t0 = time.time()
    base = pd.read_parquet(feat_path(f'base_{tag}'), columns=['user_id', 'wid', 'has_meta'])
    f = text_features(tag, base)
    p = POP.reindex(base.wid.values)
    cnt = {k: p[k].fillna(0).values.astype(np.float64) for k in ('n_ord', 'n_cart', 'n_click')}
    if tag.startswith('train'):
        # Leave-one-out. Training users are inside pop_train, so their own earlier events would be counted in
        # the popularity of products they touched; val/test users never are. Without this fix the training rows
        # get a small personalized boost that the validation rows do not, and Model A is no longer purely
        # f(Q, P). Subtracting the user's own events keeps popularity 'everyone else' on every row, so any
        # gain of Model B over A comes from the history features alone.
        own = base[['user_id', 'wid']].merge(own_counts(base.user_id.unique()), on=['user_id', 'wid'], how='left').fillna(0)
        assert len(own) == len(base), 'own-count join changed the number of rows'
        for k, o in (('n_ord', 'own_ord'), ('n_cart', 'own_cart'), ('n_click', 'own_click')):
            assert (own[o].values <= cnt[k]).all(), 'a user has more own events than the product count'
            cnt[k] = cnt[k] - own[o].values
        print(f'  leave-one-out applied to {(own[["own_ord", "own_cart", "own_click"]].sum(axis=1) > 0).mean():.2%} of rows')
    for c, src in (('prod_pop_ord', 'n_ord'), ('prod_pop_cart', 'n_cart'), ('prod_pop_click', 'n_click')):
        f[c] = np.log1p(cnt[src]).astype(np.float32)
    f['has_meta'] = base.has_meta.astype(np.int8).values
    out = pd.concat([base[['user_id', 'wid']].reset_index(drop=True), f[FEAT_A].reset_index(drop=True)], axis=1)
    assert len(out) == len(base) and (out.wid.values == base.wid.values).all() and (out.user_id.values == base.user_id.values).all()
    out.to_parquet(feat_path(f'featA_{tag}'), index=False, compression='zstd'); feat_save(f'featA_{tag}')
    print(f'featA_{tag}: {len(out):,} rows x {len(FEAT_A)} features ({time.time()-t0:.0f}s)')

def own_counts(users):
    """Each user's OWN event counts per (product, event type).
    Why: pop_train is built from all training users, so a training user's own history is inside the
    popularity count of the products they touched. Validation/test users are not in pop_train.
    Subtracting the own events makes popularity 'everyone except this shopper' on every row, so
    Model A stays purely non-personalized and train rows are treated like val/test rows."""
    ev = pq.read_table(f'{LOCAL}/events_train.parquet', columns=['user_id', 'wid', 'etype'])
    ev = ev.filter(pc.is_in(ev['user_id'], value_set=pa.array(users, ev.schema.field('user_id').type)))
    g = ev.group_by(['user_id', 'wid', 'etype']).aggregate([([], 'count_all')]).to_pandas()
    w = (g[g.etype.isin([1, 2, 3])].pivot_table(index=['user_id', 'wid'], columns='etype', values='count_all',
                                               aggfunc='sum', fill_value=0)
           .rename(columns={1: 'own_click', 2: 'own_cart', 3: 'own_ord'}).reset_index())
    w.columns.name = None
    for c in ('own_click', 'own_cart', 'own_ord'):
        if c not in w: w[c] = 0
    return w

for tag in SETS: build_featA(tag, FORCE)

build_featA('train_dev', force=True)     # validation stays as it was: its users are not in pop_train

featA_train_dev: found, skipping
featA_val: found, skipping
  leave-one-out applied to 0.78% of rows
featA_train_dev: 2,686,779 rows x 7 features (41s)


### 2.3 Check the Model A features
For each feature on the dev-training table: the share of NaN, its mean for positives and non-positives, the AUC over all rows, and the **mean AUC within a query** on a sample of 3,000 queries. The within-query AUC is the one that matters for ranking: it asks whether the feature puts positives above the other candidates of the same query. NaN is replaced by -1 for the AUC only.

What to expect from the earlier checks: the candidates were retrieved by the query, so `query_coverage` and `tfidf_cosine` should separate positives only a little (coverage 0.725 vs 0.718 in the probe); popularity should be the strongest, with a global AUC of about 0.64-0.69.

In [ ]:
from sklearn.metrics import roc_auc_score
fa = pd.read_parquet(feat_path('featA_train_dev'))
lab = pd.read_parquet(feat_path('base_train_dev'), columns=['label'])
assert len(fa) == len(lab)
fa['pos'] = (lab.label.values > 0).astype(int)
smp = np.random.default_rng(SEED).choice(fa.user_id.unique(), min(3000, fa.user_id.nunique()), replace=False)
fs = fa[fa.user_id.isin(smp)]
rows = []
for c in FEAT_A:
    wq = []
    for _, g in fs.groupby('user_id'):
        if 0 < g.pos.sum() < len(g) and g[c].nunique(dropna=False) > 1: wq.append(roc_auc_score(g.pos, g[c].fillna(-1)))
    rows.append(dict(feature=c, nan_share=round(fa[c].isna().mean(), 3), mean_pos=round(fa.loc[fa.pos == 1, c].mean(), 3),
                     mean_neg=round(fa.loc[fa.pos == 0, c].mean(), 3), auc_all=round(roc_auc_score(fa.pos, fa[c].fillna(-1)), 3),
                     auc_within_query=round(float(np.mean(wq)), 3) if wq else np.nan))
display(pd.DataFrame(rows).set_index('feature'))
print('rows:', f'{len(fa):,}', '| all finite:', bool(np.isfinite(fa[FEAT_A].fillna(0).values).all()))
del fa, fs

,nan_share,mean_pos,mean_neg,auc_all,auc_within_query
feature,,,,,
query_coverage,0.005,0.723,0.720,0.500,0.518
tfidf_cosine,0.005,0.277,0.260,0.529,0.535
brand_name_match,0.057,0.057,0.048,0.517,0.524
prod_pop_ord,0.000,1.104,0.367,0.636,0.717
prod_pop_cart,0.000,1.721,0.642,0.673,0.758
prod_pop_click,0.000,1.596,0.554,0.668,0.752
has_meta,0.000,0.998,0.995,0.501,0.509


rows: 2,686,779 | all finite: True


## 3. Model B features: f(Q, P, H_u)
Model B receives the seven Model A columns plus 23 history features. Every history feature is built from the shopper's own events (all before the query), so it can differ between candidates of one query only through the candidate. Five groups:

| Group | Features | Count | Meaning |
|---|---|---|---|
| User size | `n_ord`, `n_cart`, `n_click` | 3 | how many events of each type the shopper has (log1p); how much the shares below rest on |
| Same product | `prev_n_ord`, `prev_n_cart`, `prev_n_click`, `prev_days_since` | 4 | has the shopper already ordered / carted / clicked this exact candidate, and how long ago |
| Candidate affinity | `cat4_share_{ord,cart,click}`, `cat4_days_since_{ord,cart,click}`, `brand_share_{ord,cart,click}`, `brand_days_since_ord`, `shop_share_ord`, `deepest_match_level` | 12 | how much of the shopper's history lies in the candidate's category, brand and shop, and how recently |
| Past-query intent | `pqs_max`, `pqs_decay` | 2 | has the shopper searched for something like this candidate before (best TF-IDF similarity of the name to the shopper's past queries; the second weights each query by its age) |
| Collaborative filtering | `cf_max`, `cf_score` | 2 | how close the candidate is to the shopper's products in the CF space (highest and recency-weighted mean cosine) |

**Rules for every feature**
- **NaN is not 0.** A share is NaN if the shopper has no events of that type (the click window covers only about 7 days, so this is common), and 0 if they have events but none match. A candidate attribute that is unknown (no brand, no metadata) also gives NaN.
- **Event types are kept separate in the count and affinity features.** The observed logging horizons are approximately 365 days for orders, 90 for carts and 7 for clicks; rare events outside these horizons are retained.
- **No other users' data except CF.** Only the CF vectors are learned from other shoppers. To keep a shopper from being scored with vectors that contain their own history, CF uses five out-of-fold books (see 3.6).
- **One file per group** (`featB_<group>_<tag>`), same rows and order as `base_<tag>`, assembled into `featB_hist_<tag>` in 3.7. A group can be rebuilt without touching the others, and dropping a group is a column selection.

**Build order:** 3.1 enriched history, 3.2 user size, 3.3 same product, 3.4 candidate affinity, 3.5 past-query intent, 3.6 CF, 3.7 assembly and checks.

### 3.1 Enriched history
Builds `hist_<tag>`: the history events of the users in the table, restricted to clicks, carts and orders (follows are not used), each joined with the product's category levels, brand and shop (`products_clean`). `days` is the time between the event and the query, in days. `qless` marks events that carry no query. All other groups read this table.

The printout checks the logging windows: the largest `days` per type should be about 7 for clicks, 90 for carts and 365 for orders.

In [ ]:
HL = {1: 2.0, 2: 38.0, 3: 154.0}              # decay half-lives in days (median ages from the A4 trace): click, cart, order
ETW = {1: 1.0, 2: 2.0, 3: 3.0}                # interaction weights: click, cart, order
N_PQ, CF_DIM, CF_MIN_USERS = 50, 64, 5        # past queries kept per user, CF dimensions, minimum fitting users per product
ensure_local(['events_val', 'events_qry_train', 'events_qry_val'])      # events_train was copied in 1.5
HIST_COLS = ['cate_id_1', 'cate_id_2', 'cate_id_3', 'cate_id_4', 'brand_id', 'shop_id']

def build_hist(tag, split, user_ids, force=False):
    name = f'hist_{tag}'
    if not force and feat_exists(name): print(f'{name}: found, skipping'); return
    t0 = time.time()
    ev = pq.read_table(f'{LOCAL}/events_{split}.parquet', columns=['user_id', 'pos', 'wid', 'etype', 'qless', 'secs_before_q'])
    keep = pc.and_(pc.is_in(ev['user_id'], value_set=pa.array(user_ids, ev.schema.field('user_id').type)),
                   pc.is_in(ev['etype'], value_set=pa.array([1, 2, 3], ev.schema.field('etype').type)))
    ev = ev.filter(keep).to_pandas()
    assert (ev.secs_before_q >= 0).all(), 'an event lies after the query'
    prod = fetch_products(ev.wid.unique(), HIST_COLS, feat_path('products_clean')).to_pandas()
    h = ev.merge(prod, on='wid', how='left')
    h['days'] = (h.secs_before_q / 86400).astype('float32')
    h = h.drop(columns='secs_before_q').sort_values(['user_id', 'pos'], kind='stable').reset_index(drop=True)
    h.to_parquet(feat_path(name), index=False, compression='zstd'); feat_save(name)
    print(f'{name}: {len(h):,} events of {h.user_id.nunique():,} users ({time.time()-t0:.0f}s)')

for tag, (split, ids) in SETS.items(): build_hist(tag, split, ids, FORCE)

for tag in SETS:
    h = pd.read_parquet(feat_path(f'hist_{tag}'), columns=['etype', 'days', 'qless', 'cate_id_4', 'brand_id', 'shop_id'])
    t = h.groupby('etype').agg(events=('days', 'size'), max_days=('days', 'max'), median_days=('days', 'median'))
    t.index = t.index.map({1: 'click', 2: 'cart', 3: 'order'})
    print(f'{tag}: query-less share {h.qless.mean():.1%} | no category {h.cate_id_4.isna().mean():.1%} | no brand {h.brand_id.isna().mean():.1%} | no shop {h.shop_id.isna().mean():.1%}')
    display(t.round(2))
del h, t

hist_train_dev: found, skipping
hist_val: found, skipping
train_dev: query-less share 64.7% | no category 4.9% | no brand 6.5% | no shop 5.2%


,events,max_days,median_days
etype,,,
click,1430080,7.000000,2.780000
cart,1983329,152.649994,38.150002
order,1184851,364.970001,155.029999


val: query-less share 64.7% | no category 4.8% | no brand 6.4% | no shop 5.2%


,events,max_days,median_days
etype,,,
click,1204869,7.000000,2.820000
cart,1681255,635.289978,38.279999
order,1000813,364.970001,153.470001


### 3.2 User size
`n_ord`, `n_cart`, `n_click`: `log1p` of the number of the shopper's events of each type. The same for all candidates of a query, so the trees can use it only together with candidate features (it tells how much weight the shares deserve). A 0 here is a real count: the shopper has no events of that type in its window.

In [ ]:
FEAT_B = {
    'size': ['n_ord', 'n_cart', 'n_click'],
    'prev': ['prev_n_ord', 'prev_n_cart', 'prev_n_click', 'prev_days_since'],
    'aff':  [f'cat4_share_{s}' for s in ('ord', 'cart', 'click')] + [f'cat4_days_since_{s}' for s in ('ord', 'cart', 'click')] +
            [f'brand_share_{s}' for s in ('ord', 'cart', 'click')] + ['brand_days_since_ord', 'shop_share_ord', 'deepest_match_level'],
    'pqs':  ['pqs_max', 'pqs_decay'],
    'cf':   ['cf_max', 'cf_score'],        # cf_covered dropped: it only says "at least 5 shoppers touched this product", i.e. popularity,
                                           # which Model A already has through prod_pop_*
}
assert sum(len(v) for v in FEAT_B.values()) == 23

def load_base(tag, cols=()):
    return pd.read_parquet(feat_path(f'base_{tag}'), columns=['user_id', 'wid'] + list(cols))

def run_group(name, fn, tag, split, force=False):
    """Builds one feature group; the file has the rows of base_<tag> in the same order."""
    out_name = f'featB_{name}_{tag}'
    if not force and feat_exists(out_name): print(f'{out_name}: found, skipping'); return
    t0 = time.time()
    base, f = fn(tag, split)
    f = f[FEAT_B[name]].astype('float32')
    out = pd.concat([base[['user_id', 'wid']].reset_index(drop=True), f.reset_index(drop=True)], axis=1)
    assert len(out) == len(base) and (out.wid.values == base.wid.values).all() and (out.user_id.values == base.user_id.values).all()
    out.to_parquet(feat_path(out_name), index=False, compression='zstd'); feat_save(out_name)
    print(f'{out_name}: {len(out):,} rows x {len(FEAT_B[name])} features ({time.time()-t0:.0f}s)')

def feat_size(tag, split):
    base = load_base(tag)
    h = pd.read_parquet(feat_path(f'hist_{tag}'), columns=['user_id', 'etype'])
    c = h.groupby(['user_id', 'etype']).size().unstack(fill_value=0).reindex(columns=[3, 2, 1], fill_value=0)
    c.columns = FEAT_B['size']
    return base, np.log1p(c.reindex(base.user_id.values).fillna(0))

for tag, (split, ids) in SETS.items(): run_group('size', feat_size, tag, split, FORCE)

featB_size_train_dev: found, skipping
featB_size_val: found, skipping


### 3.3 Same product in history
For each candidate: how many times the shopper has ordered, carted and clicked exactly this product (`prev_n_*`, a real 0 if never), and the days since the last touch of any type (`prev_days_since`, NaN if never). Only 0.78% of candidates are in the history, so these fire rarely, but when they do the positive rate is about 26% against 1.9% (lift 12x). The trees weigh the three counts themselves, so there is no hand-made score.

In [ ]:
def feat_prev(tag, split):
    base = load_base(tag)
    h = pd.read_parquet(feat_path(f'hist_{tag}'), columns=['user_id', 'wid', 'etype', 'days'])
    n = h.groupby(['user_id', 'wid', 'etype']).size().unstack(fill_value=0).reindex(columns=[3, 2, 1], fill_value=0)
    n.columns = FEAT_B['prev'][:3]
    d = h.groupby(['user_id', 'wid']).days.min().rename('prev_days_since')
    key = pd.MultiIndex.from_arrays([base.user_id.values, base.wid.values])
    f = n.reindex(key).fillna(0).reset_index(drop=True)
    f['prev_days_since'] = d.reindex(key).values
    return base, f

for tag, (split, ids) in SETS.items(): run_group('prev', feat_prev, tag, split, FORCE)

featB_prev_train_dev: found, skipping
featB_prev_val: found, skipping


### 3.4 Candidate affinity
How much of the shopper's history lies in the same place as the candidate. For a key (leaf category `cate_id_4`, brand, shop) and a signal (orders, carts, clicks):

- **share** = the shopper's events of that signal with the candidate's key / all the shopper's events of that signal. NaN if the shopper has no events of the signal; 0 if they have events but none with this key; NaN if the candidate's key is unknown.
- **days_since** = days since the most recent matching event of that signal, NaN if none.

`deepest_match_level` is the deepest category level (1 = top, 4 = leaf) at which the candidate shares a category with any of the shopper's events (0 = none, NaN if the candidate's category or the shopper's history is unknown). Brand shares are computed separately for orders, carts and clicks. Brand recency and shop share are computed for orders only.

In [ ]:
def share_days(base, h, key, etype):
    hs = h[h.etype == etype]
    tot = hs.groupby('user_id').size().reindex(base.user_id.values).values.astype(float)       # NaN: no events of this signal
    cnt = hs.dropna(subset=[key]).groupby(['user_id', key]).days.agg(['size', 'min']).reset_index()
    m = base[['user_id', key]].merge(cnt, on=['user_id', key], how='left')
    assert len(m) == len(base), 'merge changed the number of rows'
    share = np.where(np.isnan(tot), np.nan, m['size'].fillna(0).values / np.where(np.isnan(tot), 1, tot))
    share[base[key].isna().values] = np.nan                                                    # unknown candidate attribute
    return share, m['min'].values

def feat_aff(tag, split):
    base = load_base(tag, ['cate_id_1', 'cate_id_2', 'cate_id_3', 'cate_id_4', 'brand_id', 'shop_id'])
    h = pd.read_parquet(feat_path(f'hist_{tag}'), columns=['user_id', 'etype', 'days'] + HIST_COLS)
    f = {}
    for s, et in (('ord', 3), ('cart', 2), ('click', 1)):
        f[f'cat4_share_{s}'], f[f'cat4_days_since_{s}'] = share_days(base, h, 'cate_id_4', et)
        f[f'brand_share_{s}'], bd = share_days(base, h, 'brand_id', et)
        if s == 'ord': f['brand_days_since_ord'] = bd
    f['shop_share_ord'], _ = share_days(base, h, 'shop_id', 3)
    # deepest category level shared with any event of the shopper
    lvl = np.zeros(len(base), np.float32)
    for k in (1, 2, 3, 4):
        col = f'cate_id_{k}'
        seen = pd.MultiIndex.from_frame(h[['user_id', col]].dropna().drop_duplicates())
        hit = pd.MultiIndex.from_arrays([base.user_id.values, base[col].values]).isin(seen) & base[col].notna().values
        lvl = np.where(hit, k, lvl)
    no_info = base.cate_id_1.isna().values | ~base.user_id.isin(h.user_id.unique()).values
    f['deepest_match_level'] = np.where(no_info, np.nan, lvl)
    return base, pd.DataFrame(f)

for tag, (split, ids) in SETS.items(): run_group('aff', feat_aff, tag, split, FORCE)

featB_aff_train_dev: found, skipping
featB_aff_val: found, skipping


### 3.5 Past-query intent
Has the shopper searched for something like this candidate before? Only history events that carry a query are used (about 35% of events), reduced to the shopper's distinct past queries, at most the 50 most recent per shopper.

- `pqs_max` = highest TF-IDF cosine between the candidate name and any past query (same IDF as `tfidf_cosine`).
- `pqs_decay` = the same, but each past query's similarity is multiplied by `0.5^(age / half-life)` first; the half-life depends on the event type (click 2, cart 38, order 154 days), and a query seen in several events keeps its freshest weight. Old queries then count less.

NaN if the shopper has no past query or the candidate has no name. No test query equals a past query (checked in the EDA), so this measures similarity, not repeats. Computed per shopper as one small matrix product (candidates x past queries).

In [ ]:
HL_ARR = np.array([np.nan, HL[1], HL[2], HL[3]])        # indexed by event type

def term_space(cols):
    """Vocabulary = training IDF terms + terms only seen here (maximum IDF), and the IDF vector."""
    here = pc.unique(pa.concat_arrays([pc.list_flatten(c) for c in cols]))
    extra = pc.filter(here, pc.invert(pc.is_in(here, value_set=IDF_VOCAB)))
    vocab = pa.concat_arrays([IDF_VOCAB, extra.cast(pa.string())])
    idf = np.concatenate([idf_tab.idf.values, np.full(len(extra), IDF_MAX)]).astype(np.float32)
    return vocab, idf

def feat_pqs(tag, split):
    base = load_base(tag)
    h = pd.read_parquet(feat_path(f'hist_{tag}'), columns=['user_id', 'pos', 'etype', 'days', 'qless'])
    h = h[~h.qless]
    qt = pq.read_table(f'{LOCAL}/events_qry_{split}.parquet')
    qt = qt.filter(pc.is_in(qt['user_id'], value_set=pa.array(h.user_id.unique(), qt.schema.field('user_id').type))).to_pandas()
    h = h.merge(qt, on=['user_id', 'pos'], how='inner')
    h = h[h.qry.notna() & (h.qry != '') & (h.qry != '-1')].copy()
    h['w'] = 0.5 ** (h.days.values / HL_ARR[h.etype.values])
    g = h.groupby(['user_id', 'qry']).agg(days=('days', 'min'), w=('w', 'max')).reset_index()          # distinct queries per user
    g = g.sort_values(['user_id', 'days'], kind='stable').groupby('user_id').head(N_PQ).sort_values('user_id', kind='stable').reset_index(drop=True)
    print(f'  N_PQ = {N_PQ} | shoppers at the cap: {(g.groupby("user_id").size() == N_PQ).mean():.1%}')

    ct = pq.read_table(feat_path(f'cand_terms_{tag}'), columns=['wid', 'terms'])
    names = ct['terms'].combine_chunks()
    qs = pc.split_pattern(pa.array(g.qry.values, pa.string()), pattern=SEP)
    vocab, idf = term_space([names, qs])
    Pt = normalize(csr_from_lists(names, vocab, len(vocab)).multiply(idf).tocsr())      # candidate names, TF-IDF, unit length
    Hq = normalize(csr_from_lists(qs, vocab, len(vocab)).multiply(idf).tocsr())         # past queries
    pi = pd.Index(ct['wid'].to_numpy()).get_indexer(base.wid.values)

    ub, r0 = np.unique(base.user_id.values, return_index=True)
    assert (np.diff(base.user_id.values) >= 0).all(), 'base rows of one user must be contiguous'
    r1 = np.r_[r0[1:], len(base)]
    uq, wq = g.user_id.values, g.w.values
    q0, q1 = np.searchsorted(uq, ub, 'left'), np.searchsorted(uq, ub, 'right')
    pmax, pdec = (np.full(len(base), np.nan, np.float32) for _ in range(2))
    for a, b, c0, c1 in zip(r0, r1, q0, q1):
        if c1 == c0: continue                                                           # no past query: NaN
        rows = pi[a:b]; ok = rows >= 0
        S = (Pt[np.where(ok, rows, 0)] @ Hq[c0:c1].T).toarray()
        S[~ok] = np.nan                                                                 # candidate without a name
        pmax[a:b], pdec[a:b] = S.max(1), (S * wq[c0:c1]).max(1)
    return base, pd.DataFrame({'pqs_max': pmax, 'pqs_decay': pdec})

for tag, (split, ids) in SETS.items(): run_group('pqs', feat_pqs, tag, split, True)

  N_PQ = 50 | shoppers at the cap: 18.5%
featB_pqs_train_dev: 2,686,779 rows x 2 features (83s)
  N_PQ = 50 | shoppers at the cap: 18.1%
featB_pqs_val: 2,294,519 rows x 2 features (66s)


### 3.6 Collaborative filtering with out-of-fold books
**What a book is.** A set of 64-number vectors, one per product, learned from which shoppers touched which products: weighted interactions (order 3, cart 2, click 1, then `log1p`) in a product x shopper matrix, reduced by a truncated SVD. Products touched by similar shoppers get similar vectors. Only products touched by at least 5 different fitting shoppers get a vector.

**Why folds.** If the vectors were fitted on a shopper's own history and then used to score that shopper, the shopper would help write the entries used to score them, and training rows would look easier than validation rows. So the 5 user folds from `users.parquet` define 5 books: book *k* is fitted on all training shoppers **not** in fold *k*, and shoppers in fold *k* are scored with book *k*. Validation and test shoppers are scored with a sixth book fitted on all training shoppers. Every row, in training, validation and test, then meets vectors that never saw the shopper's own history.

**Features** (the candidate's vector against the vectors of the shopper's products, cosine; the candidate itself is excluded from the shopper's products):
- `cf_max` = the highest cosine; `cf_score` = the mean cosine weighted by interaction weight x recency decay.
- `cf_max` and `cf_score` are NaN if the candidate has no vector or none of the shopper's other history products has one. `cf_covered` is computed internally but is not included in Model B.

The books are built once (about 3 to 5 minutes each) and saved, and the same code scores the 30k development shoppers now and all shoppers in section 4.

In [ ]:
from sklearn.decomposition import TruncatedSVD

def build_cf_books(force=False):
    names = [f'cf_book_{b}' for b in list(range(N_FOLDS)) + ['full']]
    if not force and all(feat_exists(n, 'npz') for n in names): print('cf books: found, skipping'); return
    t0 = time.time()
    ev = pq.read_table(f'{LOCAL}/events_train.parquet', columns=['user_id', 'wid', 'etype'])
    ev = ev.filter(pc.is_in(ev['etype'], value_set=pa.array([1, 2, 3], ev.schema.field('etype').type)))
    w = pc.if_else(pc.equal(ev['etype'], 3), 3.0, pc.if_else(pc.equal(ev['etype'], 2), 2.0, 1.0))
    ui = pa.table({'user_id': ev['user_id'], 'wid': ev['wid'], 'w': w}).group_by(['user_id', 'wid']).aggregate([('w', 'sum')]).to_pandas()
    del ev, w
    fold_by_uid = np.full(users.user_id.max() + 1, -1, np.int16)
    fold_by_uid[users.user_id.values] = np.where(users.split.values == 'train', users.fold.values, -1)
    fr = fold_by_uid[ui.user_id.values]
    print(f'interactions of training users: {len(ui):,} (user, product) pairs')
    for b in list(range(N_FOLDS)) + ['full']:
        tb = time.time()
        sub = ui[(fr >= 0) if b == 'full' else ((fr >= 0) & (fr != b))]
        keep = sub.wid.value_counts()
        sub = sub[sub.wid.isin(keep.index[keep >= CF_MIN_USERS])]
        icode, items = pd.factorize(sub.wid.values)
        ucode, _ = pd.factorize(sub.user_id.values)
        X = sp.csr_matrix((np.log1p(sub.w_sum.values).astype(np.float32), (icode, ucode)), shape=(len(items), ucode.max() + 1))
        k = int(min(CF_DIM, X.shape[0] - 1, X.shape[1] - 1))
        E = normalize(TruncatedSVD(k, algorithm='randomized', n_iter=5, random_state=SEED).fit_transform(X))
        np.savez_compressed(feat_path(f'cf_book_{b}', 'npz'), wid=np.asarray(items, np.int64), vec=E.astype(np.float16))
        feat_save(f'cf_book_{b}', 'npz')
        print(f'book {b}: fitted on {X.shape[1]:,} shoppers, {X.shape[0]:,} products with a vector, {k} dimensions ({time.time()-tb:.0f}s)')
    print(f'cf books done in {time.time()-t0:.0f}s')

build_cf_books(FORCE)

cf books: found, skipping


In [ ]:
def load_book(b):
    z = np.load(feat_path(f'cf_book_{b}', 'npz'))
    return pd.Index(z['wid']), z['vec'].astype(np.float32)

def feat_cf(tag, split):
    base = load_base(tag)
    h = pd.read_parquet(feat_path(f'hist_{tag}'), columns=['user_id', 'wid', 'etype', 'days']).sort_values('user_id', kind='stable')
    assert (np.diff(base.user_id.values) >= 0).all(), 'base rows of one user must be contiguous'
    ub, r0 = np.unique(base.user_id.values, return_index=True); r1 = np.r_[r0[1:], len(base)]
    h0, h1 = np.searchsorted(h.user_id.values, ub, 'left'), np.searchsorted(h.user_id.values, ub, 'right')
    # book of each user: the fold of a training user, the full book for validation and test users
    ufold = users.set_index('user_id').fold.reindex(ub).values
    book_of = np.where((split == 'train') & (ufold >= 0), ufold, -1)
    hw, het, hd = h.wid.values, h.etype.values.astype(int), h.days.values
    bw = base.wid.values
    cmax, cscore, ccov = (np.full(len(base), np.nan, np.float32) for _ in range(3))
    for bk in np.unique(book_of):
        idx, V = load_book('full' if bk < 0 else int(bk))
        for j in np.where(book_of == bk)[0]:
            a, b, c0, c1 = r0[j], r1[j], h0[j], h1[j]
            cand = idx.get_indexer(bw[a:b]); inb = cand >= 0
            ccov[a:b] = inb
            hi = idx.get_indexer(hw[c0:c1]); kp = hi >= 0
            if not inb.any() or not kp.any(): continue
            S = V[np.where(inb, cand, 0)] @ V[hi[kp]].T                       # candidates x the shopper's products (cosine)
            S[bw[a:b][:, None] == hw[c0:c1][kp][None, :]] = np.nan           # a product is not compared with itself
            wt = np.array([ETW[e] for e in het[c0:c1][kp]]) * 0.5 ** (hd[c0:c1][kp] / HL_ARR[het[c0:c1][kp]])
            ok = ~np.isnan(S)
            den = (ok * wt).sum(1)
            with np.errstate(all='ignore'):
                mx = np.where(ok.any(1), np.where(ok, S, -np.inf).max(1), np.nan)
                sc = np.where(den > 0, np.where(ok, S, 0) @ wt / den, np.nan)
            cmax[a:b], cscore[a:b] = np.where(inb, mx, np.nan), np.where(inb, sc, np.nan)
    return base, pd.DataFrame({'cf_max': cmax, 'cf_score': cscore, 'cf_covered': ccov})

for tag, (split, ids) in SETS.items(): run_group('cf', feat_cf, tag, split, FORCE)

featB_cf_train_dev: found, skipping
featB_cf_val: found, skipping


### 3.7 Assemble `featB_hist_<tag>` and check the history features
Joins the five groups (same rows, same order as `base_<tag>`) into one file with the 23 history columns. Model B is then `featA_<tag>` + `featB_hist_<tag>` joined on (user_id, wid).

The check table compares dev-training with validation for each feature: the share of NaN and the **mean AUC within a query** (3,000-query sample each). For `*_days_since` features a smaller value means more recent, so the table scores their negative (NaN = never = lowest). What to look for:
- training and validation should look alike (NaN share and AUC within a few points); a gap, above all for the CF features, would mean a leak between the CF books and the rows;
- the direct-hit features (`prev_*`) should show a high positive rate when present;
- the groups with the strongest signal tell us where the personalization gain will come from; AUC near 0.5 means the feature adds nothing on its own.

In [ ]:
def assemble_B(tag, force=False):
    name = f'featB_hist_{tag}'
    if not force and feat_exists(name): print(f'{name}: found, skipping'); return
    parts = [pd.read_parquet(feat_path(f'featB_{g}_{tag}'), columns=['user_id', 'wid'] + FEAT_B[g]) for g in FEAT_B]
    key = parts[0][['user_id', 'wid']]
    for p in parts[1:]: assert (p.user_id.values == key.user_id.values).all() and (p.wid.values == key.wid.values).all()
    out = pd.concat([key] + [p.drop(columns=['user_id', 'wid']) for p in parts], axis=1)
    assert out.shape[1] == 2 + 23
    out.to_parquet(feat_path(name), index=False, compression='zstd'); feat_save(name)
    print(f'{name}: {len(out):,} rows x 23 history features')

for tag in SETS: assemble_B(tag, FORCE)

from sklearn.metrics import roc_auc_score
ALL_B = [c for g in FEAT_B.values() for c in g]
def within_auc(tag, n_users=3000):
    fb = pd.read_parquet(feat_path(f'featB_hist_{tag}'))
    fb['pos'] = (pd.read_parquet(feat_path(f'base_{tag}'), columns=['label']).label.values > 0).astype(int)
    smp = np.random.default_rng(SEED).choice(fb.user_id.unique(), min(n_users, fb.user_id.nunique()), replace=False)
    fs = fb[fb.user_id.isin(smp)]
    res = {}
    for c in ALL_B:
        x = -fs[c].fillna(1e4) if c.endswith('days_since') or c == 'prev_days_since' else fs[c].fillna(-1)
        a = []
        for _, idx in fs.groupby('user_id').indices.items():
            y = fs.pos.values[idx]
            if 0 < y.sum() < len(y) and len(np.unique(x.values[idx])) > 1: a.append(roc_auc_score(y, x.values[idx]))
        res[c] = (fb[c].isna().mean(), float(np.mean(a)) if a else np.nan)
    return res, fb

res_tr, fb = within_auc('train_dev')
res_va, _ = within_auc('val')
tab = pd.DataFrame({'nan_train': {c: v[0] for c, v in res_tr.items()}, 'nan_val': {c: v[0] for c, v in res_va.items()},
                    'auc_train': {c: v[1] for c, v in res_tr.items()}, 'auc_val': {c: v[1] for c, v in res_va.items()}}).round(3)
display(tab)
hit = fb.prev_days_since.notna()
print(f'rows with the candidate already in the history: {hit.mean():.2%} | positive rate {fb.pos[hit].mean():.1%} vs {fb.pos[~hit].mean():.1%} otherwise (EDA: 25.9% vs 1.9%)')
del fb

featB_hist_train_dev: found, skipping
featB_hist_val: found, skipping


,nan_train,nan_val,auc_train,auc_val
n_ord,0.000,0.000,NaN,NaN
n_cart,0.000,0.000,NaN,NaN
n_click,0.000,0.000,NaN,NaN
prev_n_ord,0.000,0.000,0.666,0.669
prev_n_cart,0.000,0.000,0.651,0.648
prev_n_click,0.000,0.000,0.697,0.676
prev_days_since,0.992,0.992,0.698,0.687
cat4_share_ord,0.087,0.092,0.535,0.539
cat4_share_cart,0.047,0.049,0.559,0.567
cat4_share_click,0.169,0.168,0.571,0.579


rows with the candidate already in the history: 0.78% | positive rate 25.9% vs 1.9% otherwise (EDA: 25.9% vs 1.9%)


## 4. Full-data build: all training users and the test set
Builds the same files as sections 1 to 3 for **all usable training users** (about 120k) and for the **test users**, using the same functions. Nothing new is computed: the shared statistics (`pop_train`, `idf_train`) and the CF books already cover all training users.

**How:** the training users are split into `N_CHUNKS` chunks of about 30k users (the size of the development run, so memory stays as it was). Each chunk is a tag (`train_c0` ... `train_c3`) and gets `base_`, `featA_` and `featB_hist_` files. The test users are one more tag, `test`. The test features are built here but **not used** until the final evaluation in File 3.

**Per chunk** the cell builds the base table, the Model A features, the enriched history, the five history groups and the assembly, then deletes the large intermediate files (term lists, history, per-group files) so Colab and Drive do not fill up. Only `base_`, `featA_` and `featB_hist_` per tag remain. A rerun skips every chunk whose three final files exist, so the build can be stopped and resumed. Expect roughly 5 to 10 minutes per chunk, so about 40 to 60 minutes in total. If a chunk was interrupted while writing a file, rerun it with `FORCE = True` (this rebuilds all chunks).

In [ ]:
N_CHUNKS = 4                 # about 30k training users per chunk
BUILD_TEST = True            # test features are built here, but only used for the final evaluation
ensure_local(['events_test', 'events_qry_test'])

train_ids = np.sort(users.loc[(users.split == 'train') & users.use, 'user_id'].values)
FULL_SETS = {f'train_c{i}': ('train', ch) for i, ch in enumerate(np.array_split(train_ids, N_CHUNKS))}
if BUILD_TEST: FULL_SETS['test'] = ('test', np.sort(users.loc[(users.split == 'test') & users.use, 'user_id'].values))
FINAL = lambda tag: [f'base_{tag}', f'featA_{tag}', f'featB_hist_{tag}']
INTERMEDIATE = lambda tag: [f'cand_terms_{tag}', f'qry_terms_{tag}', f'hist_{tag}'] + [f'featB_{g}_{tag}' for g in FEAT_B]

def build_all(tag, split, ids, force=False):
    if not force and all(feat_exists(n) for n in FINAL(tag)): print(f'{tag}: complete, skipping'); return
    t0 = time.time(); print(f'--- {tag}: {len(ids):,} users')
    build_tag(tag, split, ids, force)
    build_featA(tag, force)
    build_hist(tag, split, ids, force)
    for gname, fn in (('size', feat_size), ('prev', feat_prev), ('aff', feat_aff), ('pqs', feat_pqs), ('cf', feat_cf)):
        run_group(gname, fn, tag, split, force)
    assemble_B(tag, force)
    print(f'{tag} done in {(time.time()-t0)/60:.1f} min')

def drop_intermediates(tag):
    """Removes the large intermediate files locally and on Drive once the three final files exist."""
    if not all(os.path.exists(feat_path(n)) for n in FINAL(tag)): return
    for n in INTERMEDIATE(tag):
        for p in (feat_path(n), f'{FEAT}/{n}.parquet'):
            if os.path.exists(p): os.remove(p)

for tag, (split, ids) in FULL_SETS.items():
    build_all(tag, split, ids, FORCE)
    drop_intermediates(tag)
print('full build finished')

train_c0: complete, skipping
train_c1: complete, skipping
train_c2: complete, skipping
train_c3: complete, skipping
test: complete, skipping
full build finished


### 4.1 Check the full build
For every chunk and the test set: the user set equals the requested users, the three files line up row by row, and the NaN share of every feature is about the same as in the development run (`train_dev`). A feature whose NaN share differs a lot between chunks, or between a chunk and the test set, points to a problem in that chunk. The totals should be about 120,174 usable training users and 25,767 usable test users.

In [ ]:
rows, nan = [], {}
for tag, (split, ids) in FULL_SETS.items():
    b = load_base(tag)
    fa = pd.read_parquet(feat_path(f'featA_{tag}'))
    fb = pd.read_parquet(feat_path(f'featB_hist_{tag}'))
    for x in (fa, fb): assert len(x) == len(b) and (x.user_id.values == b.user_id.values).all() and (x.wid.values == b.wid.values).all(), f'{tag}: files are not aligned'
    assert set(b.user_id) == set(ids), f'{tag}: user set differs from the requested users'
    rows.append(dict(table=tag, users=b.user_id.nunique(), rows=len(b), featA_cols=fa.shape[1] - 2, featB_cols=fb.shape[1] - 2))
    nan[tag] = pd.concat([fa.drop(columns=['user_id', 'wid']), fb.drop(columns=['user_id', 'wid'])], axis=1).isna().mean()
    del b, fa, fb
dev = pd.concat([pd.read_parquet(feat_path('featA_train_dev')).drop(columns=['user_id', 'wid']),
                 pd.read_parquet(feat_path('featB_hist_train_dev')).drop(columns=['user_id', 'wid'])], axis=1).isna().mean()
summary = pd.DataFrame(rows).set_index('table'); display(summary)
tr = summary.index.str.startswith('train')
print(f'training users: {summary.users[tr].sum():,} (usable in the data: {len(train_ids):,}) | rows {summary.rows[tr].sum():,}')
if 'test' in summary.index: print(f'test users: {summary.users["test"]:,}')
display(pd.DataFrame({'train_dev': dev, **nan}).round(3))

,users,rows,featA_cols,featB_cols
table,,,,
train_c0,30044,2685920,7,23
train_c1,30044,2681029,7,23
train_c2,30043,2687133,7,23
train_c3,30043,2696030,7,23
test,25767,2304086,7,23


training users: 120,174 (usable in the data: 120,174) | rows 10,750,112
test users: 25,767


,train_dev,train_c0,train_c1,train_c2,train_c3,test
query_coverage,0.005,0.005,0.004,0.005,0.005,0.004
tfidf_cosine,0.005,0.005,0.004,0.005,0.005,0.004
brand_name_match,0.057,0.058,0.057,0.058,0.059,0.058
prod_pop_ord,0.000,0.000,0.000,0.000,0.000,0.000
prod_pop_cart,0.000,0.000,0.000,0.000,0.000,0.000
prod_pop_click,0.000,0.000,0.000,0.000,0.000,0.000
has_meta,0.000,0.000,0.000,0.000,0.000,0.000
n_ord,0.000,0.000,0.000,0.000,0.000,0.000
n_cart,0.000,0.000,0.000,0.000,0.000,0.000
n_click,0.000,0.000,0.000,0.000,0.000,0.000


In [ ]:
import pyarrow.parquet as pq
from google.colab import drive

problems, rows = [], []
for tag in FULL_SETS:
    counts = {}
    for n in FINAL(tag):
        p = f'{FEAT}/{n}.parquet'
        if not os.path.exists(p) or os.path.getsize(p) == 0:
            problems.append(f'MISSING/EMPTY: {n}'); continue
        try:
            counts[n] = pq.ParquetFile(p).metadata.num_rows   # fails on a truncated file
        except Exception as e:
            problems.append(f'UNREADABLE: {n} ({e})'); continue
    if len(counts) == 3 and len(set(counts.values())) != 1:
        problems.append(f'ROW MISMATCH {tag}: {counts}')
    rows.append((tag, *counts.values()))

for dev in ('train_dev', 'val'):
    for n in (f'base_{dev}', f'featA_{dev}', f'featB_hist_{dev}'):
        if not os.path.exists(f'{FEAT}/{n}.parquet'): problems.append(f'MISSING: {n}')

if problems:
    raise RuntimeError('Feature build incomplete:\n' + '\n'.join(problems))
print(pd.DataFrame(rows, columns=['tag', 'base', 'featA', 'featB_hist']))
print('All feature files present, readable and aligned.')
drive.flush_and_unmount()   # waits for Drive to finish uploading
print('Drive flushed. Safe to close the session.')

        tag     base    featA  featB_hist
0  train_c0  2685920  2685920     2685920
1  train_c1  2681029  2681029     2681029
2  train_c2  2687133  2687133     2687133
3  train_c3  2696030  2696030     2696030
4      test  2304086  2304086     2304086
All feature files present, readable and aligned.
Drive flushed. Safe to close the session.
